In [0]:
import requests
import time
import zipfile
import shutil
import os
from pathlib import Path
from pyspark.sql import functions as F
from functools import reduce
import hashlib

# Configurações gerais
VOLUME = Path("/Volumes/workspace/bronze/raw_inep_superior")
TABELA_BRONZE = "workspace.bronze.inep_superior_cursos"

# Utilizado para forçar a reprocessamento de um periodo especifico (True) ou não  (False)
dbutils.widgets.dropdown(
    "forcar_reprocessamento",
    "false",
    ["false", "true"]
)

FORCAR_REPROCESSAMENTO = (
    dbutils.widgets.get("forcar_reprocessamento") == "true"
)

ANOS = range(2020, 2025)

print("Pipeline INEP - Landing para Bronze")
print(f"Tabela destino: {TABELA_BRONZE}")

In [0]:
# Está celular verifica os dados originais e calcula o hash SHA256 de cada um arquivo de origem
hashes_origem = {}

for ano in ANOS:

    caminho_zip = (
        VOLUME / str(ano) / f"microdados_inep_{ano}.zip"
    )

    if not caminho_zip.is_file():
        raise FileNotFoundError(
            f"ZIP não encontrado para {ano}: {caminho_zip}"
        )

    sha256 = hashlib.sha256()

    with open(caminho_zip, "rb") as arquivo:
        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b""
        ):
            sha256.update(bloco)

    hashes_origem[ano] = sha256.hexdigest()

    print(f"{ano}: ZIP identificado | SHA256: {sha256.hexdigest()[:16]}...")

print(f"\nTotal de arquivos identificados: {len(hashes_origem)}")

In [0]:
# Consulta a tabela Bronze e verifica se cada ano já foi processado com a mesma versão de origem
anos_pendentes = []

if spark.catalog.tableExists(TABELA_BRONZE):

    versoes = (
        spark.table(TABELA_BRONZE)
        .groupBy("NU_ANO_CENSO")
        .agg(
            F.count("*").alias("total_registros"),
            F.countDistinct("hash_origem").alias("qtd_versoes"),
            F.first("hash_origem").alias("hash_bronze")
        )
        .collect()
    )

    controle_bronze = {
        int(linha["NU_ANO_CENSO"]): linha.asDict()
        for linha in versoes
        if linha["NU_ANO_CENSO"] is not None
    }

else:
    controle_bronze = {}
    print("Tabela Bronze ainda não existe. Primeira carga necessária.")

for ano in ANOS:

    atual = controle_bronze.get(ano)
    hash_zip = hashes_origem[ano]

    atualizado = (
        not FORCAR_REPROCESSAMENTO
        and atual is not None
        and atual["total_registros"] > 0
        and atual["qtd_versoes"] == 1
        and atual["hash_bronze"] == hash_zip
    )

    if atualizado:
        print(f"{ano}: atualizado. Nenhuma ação necessária.")
    else:
        anos_pendentes.append(ano)
        print(f"{ano}: pendente de processamento.")

print(f"\nAnos pendentes: {anos_pendentes}")

if not anos_pendentes:
    dbutils.notebook.exit("SEM_PENDENCIAS")

In [0]:
arquivos_csv = {}

for ano in anos_pendentes:

    caminho_zip = (
        VOLUME / str(ano) / f"microdados_inep_{ano}.zip"
    )

    pasta_staging = VOLUME / "_staging" / str(ano)
    pasta_staging.mkdir(parents=True, exist_ok=True)

    destino = pasta_staging / f"microdados_cursos_{ano}.csv"
    temporario = pasta_staging / f"microdados_cursos_{ano}.csv.part"

    with zipfile.ZipFile(caminho_zip, "r") as zip_ref:

        encontrados = [
            nome for nome in zip_ref.namelist()
            if Path(nome).name.upper()
            == f"MICRODADOS_CADASTRO_CURSOS_{ano}.CSV"
        ]

        if len(encontrados) != 1:
            raise ValueError(
                f"{ano}: esperado um CSV de cursos, "
                f"encontrados {len(encontrados)}"
            )

        try:
            with zip_ref.open(encontrados[0]) as origem:
                with open(temporario, "wb") as saida:
                    shutil.copyfileobj(
                        origem,
                        saida,
                        length=1024 * 1024
                    )

            os.replace(temporario, destino)

        finally:
            if temporario.exists():
                temporario.unlink()

    arquivos_csv[ano] = str(destino)

    print(
        f"{ano}: CSV extraído | "
        f"{destino.stat().st_size / 1024**2:.2f} MiB"
    )

In [0]:
anos_atualizados = []
resumo_cargas = []

for ano in anos_pendentes:

    print(f"\n{'=' * 55}")
    print(f"Processando Bronze INEP - {ano}")
    print(f"{'=' * 55}")

    caminho_csv = arquivos_csv[ano]

    # 1. Leitura do arquivo original
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "iso-8859-1")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .csv(caminho_csv)
    )

    # 2. Validar colunas
    colunas = [c.upper() for c in df.columns]

    if len(colunas) != len(set(colunas)):
        raise ValueError(f"{ano}: colunas duplicadas.")

    if "NU_ANO_CENSO" not in df.columns:
        raise ValueError(f"{ano}: NU_ANO_CENSO ausente.")

    # 3. Validar ano dos registros
    divergentes = (
        df.filter(
            F.col("NU_ANO_CENSO").isNull()
            | (F.trim(F.col("NU_ANO_CENSO")) != str(ano))
        )
        .limit(1)
        .count()
    )

    if divergentes:
        raise ValueError(f"{ano}: ano do censo divergente.")

    total_origem = df.count()

    if total_origem == 0:
        raise ValueError(f"{ano}: CSV sem registros.")

    # 4. Metadados de rastreabilidade
    df_final = (
        df
        .withColumn("arquivo_origem", F.lit(caminho_csv))
        .withColumn("ano_ingestao", F.lit(ano))
        .withColumn("hash_origem", F.lit(hashes_origem[ano]))
        .withColumn("dt_ingestao", F.current_timestamp())
    )

    # 5. Primeira carga ou atualização seletiva
    if not spark.catalog.tableExists(TABELA_BRONZE):

        (
            df_final.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(TABELA_BRONZE)
        )

        print(f"{ano}: tabela Bronze criada.")

    else:

        (
            df_final.write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"NU_ANO_CENSO = '{ano}'"
            )
            .option("mergeSchema", "true")
            .saveAsTable(TABELA_BRONZE)
        )

        print(f"{ano}: período atualizado na Bronze.")

    anos_atualizados.append(ano)

    resumo_cargas.append({
        "ano": ano,
        "registros_origem": total_origem,
        "colunas_origem": len(df.columns)
    })

print(f"\nAnos atualizados: {anos_atualizados}")

In [0]:
df_validacao = (
    spark.table(TABELA_BRONZE)
    .groupBy("NU_ANO_CENSO")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("hash_origem").alias("versoes_origem")
    )
    .orderBy("NU_ANO_CENSO")
)

display(df_validacao)

controle_final = {
    int(linha["NU_ANO_CENSO"]): linha
    for linha in df_validacao.collect()
}

for carga in resumo_cargas:

    ano = carga["ano"]
    esperado = carga["registros_origem"]
    encontrado = controle_final.get(ano)

    if encontrado is None:
        raise ValueError(f"{ano}: ausente na Bronze.")

    if encontrado["total_registros"] != esperado:
        raise ValueError(
            f"{ano}: divergência de registros "
            f"({encontrado['total_registros']} != {esperado})."
        )

    if encontrado["versoes_origem"] != 1:
        raise ValueError(
            f"{ano}: múltiplas versões na Bronze."
        )

print("\nValidação pós-carga concluída com sucesso.")

dbutils.notebook.exit(
    "ANOS_ATUALIZADOS=" + ",".join(map(str, anos_atualizados))
)